# Coevolution with Bayesian Updates

In [1]:
import numpy as np
# CONSTANTS: defining constants for evolutionary algorithm parameters

INITIAL_CODE_POPULATION_SIZE = 10
INITIAL_TEST_POPULATION_SIZE = 20

C_0 = 0.6 # Prior probability of a code being correct
T_0 = 0.2 # Prior probability of a test being correct

# Hyperparameters for Bayesian updating, undetermisitic cases. 
ALPHA = 0.01
BETA = 0.2
GAMMA = 0.2

# Simulation constants
PROBABILITY_OF_CODE_TEST_PASSING = 0.2

In [2]:
from typing import Literal

def sign_condition(interactor: float, alpha: float, beta: float, gamma: float, updating_param: Literal["code", "test"] = "code") -> bool:
    """
    Checks if the "Intuitive Sign Condition" holds for a belief update.

    This condition determines whether an observation (pass/fail) is 
    interpreted intuitively.
    
    - For a "code" update:
      - If True, a "pass" is good news (WoE > 0) and a "fail" is 
        bad news (WoE < 0).
      - If False, the system is "perverse," and a "pass" is bad news.
    - For a "test" update:
      - If True, a "pass" is good news (WoE > 0) for the test's 
        correctness.
      - If False, a "pass" is bad news for the test's correctness.

    This is equivalent to checking if P(pass | H) > P(pass | ~H).

    Args:
        interactor: The belief (probability) in the interacting entity. 
                    If updating_param is "code", this is t = P(T_j). 
                    If "test", this is c = P(C_i).
        alpha: The probability P(pass | Correct Code, Incorrect Test).
        beta: The probability P(pass | Incorrect Code, Correct Test).
        gamma: The probability P(pass | Incorrect Code, Incorrect Test).
        updating_param: Specifies whether the belief update is for 
                        a "code" candidate or a "test" case.

    Returns:
        True if the intuitive sign condition holds, False otherwise.
    """
    if updating_param == "code":
        # Checks if t(1 - α - β + γ) > γ - α
        return (interactor * (1 - alpha - beta + gamma) > (gamma - alpha))
    elif updating_param == "test":
        # Checks if c(1 - α - β + γ) > γ - β
        return (interactor * (1 - alpha - beta + gamma) > (gamma - beta))
    else:
        raise ValueError("updating_param must be 'code' or 'test'")

def trend_condition(alpha: float, beta: float, gamma: float) -> bool:
    """
    Checks if the "Intuitive Trend Condition" holds for a "pass" observation.

    This condition determines how the Weight of Evidence (WoE) for a "pass" 
    changes as trust in the interactor (t or c) increases.

    - If True (γ > αβ), the system is in the "Intuitive Regime":
      Trusting the interactor more makes a "pass" *stronger* positive 
      evidence (WoE increases).
    - If False (γ < αβ), the system is in the "Perverse Regime":
      Trusting the interactor more paradoxically makes a "pass" *weaker* evidence (WoE decreases).

    Args:
        alpha: The probability P(pass | Correct Code, Incorrect Test).
        beta: The probability P(pass | Incorrect Code, Correct Test).
        gamma: The probability P(pass | Incorrect Code, Incorrect Test).

    Returns:
        True if the intuitive trend holds (γ > αβ), False otherwise.
    """
    return (gamma > alpha * beta)

print("Sign Condition for code updates: ", sign_condition(T_0, ALPHA, BETA, GAMMA))
print("Sign Condition for test updates: ", sign_condition(C_0, ALPHA, BETA, GAMMA, updating_param="test"))
print("Trend Condition: ", trend_condition(ALPHA, BETA, GAMMA))

Sign Condition for code updates:  True
Sign Condition for test updates:  True
Trend Condition:  True


In [3]:
import numpy as np

# Convert log-odds to probabilities
C = np.array([C_0 for _ in range(INITIAL_CODE_POPULATION_SIZE)])
T = np.array([T_0 for _ in range(INITIAL_TEST_POPULATION_SIZE)])   

print("Initial Code Population Probabilities:", C)
print("Initial Test Population Probabilities:", T)

Initial Code Population Probabilities: [0.6 0.6 0.6 0.6 0.6 0.6 0.6 0.6 0.6 0.6]
Initial Test Population Probabilities: [0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2
 0.2 0.2]


# Initial Code-Test Evaluation

In [4]:
def _evaluate_code_test_pair(code_idx: int, test_idx: int) -> int:
    # Simulation of code-test evaluation
    # Randomly return True (pass) or False (fail)
    return int(np.random.rand() < PROBABILITY_OF_CODE_TEST_PASSING)

def evaluate_population(C: np.ndarray, T: np.ndarray) -> np.ndarray:
    matrix = np.zeros((C.shape[0], T.shape[0]), dtype=np.int8)
    for i in range(C.shape[0]):
        for j in range(T.shape[0]):
            matrix[i, j] = _evaluate_code_test_pair(i, j)
    return matrix

matrix = evaluate_population(C, T)
matrix  # Convert boolean to int for better visualization

array([[0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1],
       [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1],
       [0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
       [0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1],
       [0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 1],
       [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0],
       [0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 0],
       [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0]],
      dtype=int8)

In [5]:
# matrix from an actual run
matrix = np.array([
       [0, 1, 1, 1, 1, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 0],
       [1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 1],
       [1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1],
       [0, 1, 1, 1, 1, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 0],
       [0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1],
       [0, 1, 1, 1, 1, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 0],
       [0, 1, 1, 1, 1, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 0],
       [0, 1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 0],
       [0, 1, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0]])

# Posterior Probability Update

In [6]:
def get_posterior_update_for_code(
    code_prob: float, 
    test_prob: float, 
    observation: bool, 
    alpha: float = ALPHA, 
    beta: float = BETA,
    gamma: float = GAMMA
) -> float:
    """
    Updates the probability of a code being correct based on a test result using Bayesian inference.

    This function implements the update rules derived in the 'Bayesian Coevolution of Code and Tests' article.

    Args:
        code_prob: Prior probability of the code being correct, P(C_i).
        test_prob: Probability of the test being correct, P(T_j).
        observation: Result of the test (True for pass, False for fail), D_ij.
        alpha: Hyperparameter for a correct code passing an incorrect test.
        beta: Hyperparameter for an incorrect code passing a correct test.
        gamma: Hyperparameter for an incorrect code passing an incorrect test.
    Returns:
        Updated (posterior) probability of the code being correct.
    """
    # Handle edge cases where belief is already certain
    if code_prob == 1.0 or code_prob == 0.0:
        return code_prob

    if observation:  # This is Case 1: Observation is a Pass (D_ij = 1)
        # Likelihood of a pass if code is correct: P(D=1|C=1)
        like_code_correct = test_prob + alpha * (1 - test_prob)
        
        # Likelihood of a pass if code is incorrect: P(D=1|C=0)
        like_code_incorrect = beta * test_prob + gamma * (1 - test_prob)
        
        # Denominator of the ratio in the simplified Bayes' formula
        # This corresponds to P(E|H)P(H)
        denominator_ratio = like_code_correct * code_prob
        if denominator_ratio == 0:
            return 0.0 # Evidence is impossible under this hypothesis, posterior must be 0

        # Numerator of the ratio in the simplified Bayes' formula
        # This corresponds to P(E|~H)P(~H)
        numerator_ratio = like_code_incorrect * (1 - code_prob)

        ratio = numerator_ratio / denominator_ratio
        
    else:  # This is Case 2: Observation is a Fail (D_ij = 0)
        # Likelihood of a fail if code is correct: P(D=0|C=1)
        like_code_correct = (1 - alpha) * (1 - test_prob)
        
        # Likelihood of a fail if code is incorrect: P(D=0|C=0)
        like_code_incorrect = (1 - beta) * test_prob + (1 - gamma) * (1 - test_prob)
        
        denominator_ratio = like_code_correct * code_prob
        if denominator_ratio == 0:
            return 0.0

        numerator_ratio = like_code_incorrect * (1 - code_prob)
        
        ratio = numerator_ratio / denominator_ratio
        
    posterior = 1 / (1 + ratio)
    return posterior


def get_posterior_update_for_test(
    code_prob: float, 
    test_prob: float, 
    observation: bool, 
    alpha: float = ALPHA, 
    beta: float = BETA,
    gamma: float = GAMMA
) -> float:
    """
    Updates the probability of a test being correct based on its result using Bayesian inference.

    This function implements the symmetrical update rules derived in the 'Bayesian Coevolution of Code and Tests' article.

    Args:
        code_prob: Probability of the code being correct, P(C_i).
        test_prob: Prior probability of the test being correct, P(T_j).
        observation: Result of the test (True for pass, False for fail), D_ij.
        alpha: Hyperparameter for a correct code passing an incorrect test.
        beta: Hyperparameter for an incorrect code passing a correct test.
        gamma: Hyperparameter for an incorrect code passing an incorrect test.

    Returns:
        Updated (posterior) probability of the test being correct.
    """
    # Handle edge cases where belief is already certain
    if test_prob == 1.0 or test_prob == 0.0:
        return test_prob

    if observation: # This is Case 1: Observation is a Pass (D_ij = 1)
        # Likelihood of a pass if test is correct: P(D=1|T=1)
        like_test_correct = code_prob + beta * (1 - code_prob)
        
        # Likelihood of a pass if test is incorrect: P(D=1|T=0)
        like_test_incorrect = alpha * code_prob + gamma * (1 - code_prob)
        
        denominator_ratio = like_test_correct * test_prob
        if denominator_ratio == 0:
            return 0.0

        numerator_ratio = like_test_incorrect * (1 - test_prob)
        
        ratio = numerator_ratio / denominator_ratio
        
    else: # This is Case 2: Observation is a Fail (D_ij = 0)
        # Likelihood of a fail if test is correct: P(D=0|T=1)
        like_test_correct = (1 - beta) * (1 - code_prob)
        
        # Likelihood of a fail if test is incorrect: P(D=0|T=0)
        like_test_incorrect = (1 - alpha) * code_prob + (1 - gamma) * (1 - code_prob)
        
        denominator_ratio = like_test_correct * test_prob
        if denominator_ratio == 0:
            return 0.0

        numerator_ratio = like_test_incorrect * (1 - test_prob)
        
        ratio = numerator_ratio / denominator_ratio
        
    posterior = 1 / (1 + ratio)
    return posterior
def update_all_probabilities(C_prev: np.ndarray, T_prev: np.ndarray, matrix: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """
    Demonstrates the Bayesian update for all codes and tests based on the evaluation matrix.
    Args:
        C: Array of probabilities that each code is correct
        T: Array of probabilities that each test is correct
        matrix: Evaluation matrix where each entry is 1 (pass) or 0 (fail)
    """

    C_new = np.zeros_like(C_prev)
    T_new = np.zeros_like(T_prev)


    print("=== Posterior Update ===")
    print("--- Updating Code Probabilities ---")
    for i in range(C_prev.shape[0]):
        print("Previous Code Probability:", C_prev[i])

        code_prob: float = C_prev[i]
        for j in range(T_prev.shape[0]):
            test_prob: float = T_prev[j]
            observation = bool(matrix[i, j])

            code_prob = get_posterior_update_for_code(code_prob, test_prob, observation)
            print(f"  Test {j} (Prob: {test_prob:.2f}, Observation: {observation}) -> Updated Code Prob: {code_prob:.2f}")
        
        C_new[i] = code_prob
        print("Updated Code Probability:", C_new[i])

    print("--- Updating Test Probabilities ---")
    for j in range(T_prev.shape[0]):
        print("Previous Test Probability:", T_prev[j])

        test_prob = T_prev[j]
        for i in range(C_prev.shape[0]):
            code_prob = C_prev[i]
            observation = bool(matrix[i, j])

            test_prob = get_posterior_update_for_test(code_prob, test_prob, observation)
            print(f"  Code {i} (Prob: {code_prob:2f}, Observation: {observation}) -> Updated Test Prob: {test_prob:.2f}")
        
        T_new[j] = test_prob
        print("Updated Test Probability:", T_new[j])




    return C_new, T_new
    
for i in range(1):
    print(f"=== Iteration {i+1} ===")
    C, T = update_all_probabilities(C, T, matrix)
    print("Updated Code Population Probabilities:", C)
    print("Updated Test Population Probabilities:", T)

=== Iteration 1 ===
=== Posterior Update ===
--- Updating Code Probabilities ---
Previous Code Probability: 0.6
  Test 0 (Prob: 0.20, Observation: False) -> Updated Code Prob: 0.60
  Test 1 (Prob: 0.20, Observation: True) -> Updated Code Prob: 0.61
  Test 2 (Prob: 0.20, Observation: True) -> Updated Code Prob: 0.62
  Test 3 (Prob: 0.20, Observation: True) -> Updated Code Prob: 0.63
  Test 4 (Prob: 0.20, Observation: True) -> Updated Code Prob: 0.63
  Test 5 (Prob: 0.20, Observation: False) -> Updated Code Prob: 0.63
  Test 6 (Prob: 0.20, Observation: False) -> Updated Code Prob: 0.63
  Test 7 (Prob: 0.20, Observation: False) -> Updated Code Prob: 0.63
  Test 8 (Prob: 0.20, Observation: False) -> Updated Code Prob: 0.63
  Test 9 (Prob: 0.20, Observation: True) -> Updated Code Prob: 0.63
  Test 10 (Prob: 0.20, Observation: False) -> Updated Code Prob: 0.63
  Test 11 (Prob: 0.20, Observation: True) -> Updated Code Prob: 0.64
  Test 12 (Prob: 0.20, Observation: False) -> Updated Code Prob:

# LOG Based Updates

In [7]:
import coevolution_bayesian as cb

config = cb.CoevolutionConfig(
    initial_code_population_size=INITIAL_CODE_POPULATION_SIZE,
    initial_test_population_size=INITIAL_TEST_POPULATION_SIZE,
    c0_prior=C_0,
    t0_prior=T_0,
    alpha=ALPHA,
    beta=BETA,
    gamma=GAMMA
)


C, T = cb.initialize_populations(config)
print("Initial Code Population Probabilities (from module):", C)
print("Initial Test Population Probabilities (from module):", T)

C_new, T_new = cb.update_population_beliefs(C, T, matrix, config)

print("Updated Code Population Probabilities (from module):", C_new)
print("Updated Test Population Probabilities (from module):", T_new)

ModuleNotFoundError: No module named 'common.coevolution_bayesian'

# Selection

In [ ]:
# Import from the new module structure
from common.coevolution import SelectionStrategy

# Test the available methods
print("Available selection methods:", SelectionStrategy.get_available_methods())

# Create a simple test population and fitness
test_population = np.array([[1, 2], [3, 4], [5, 6], [7, 8], [9, 10]])
test_fitness = np.array([0.2, 0.5, 0.8, 0.3, 0.6])

# Test binary tournament selection
parent1, parent2 = SelectionStrategy.select_parents(test_population, test_fitness, method="binary_tournament")
print("\nBinary Tournament Selection:")
print(f"Parent 1: {parent1}")
print(f"Parent 2: {parent2}")

# Test roulette wheel selection
parent1, parent2 = SelectionStrategy.select_parents(test_population, test_fitness, method="roulette_wheel")
print("\nRoulette Wheel Selection:")
print(f"Parent 1: {parent1}")
print(f"Parent 2: {parent2}")

# Test rank selection
parent1, parent2 = SelectionStrategy.select_parents(test_population, test_fitness, method="rank_selection")
print("\nRank Selection:")
print(f"Parent 1: {parent1}")
print(f"Parent 2: {parent2}")

# Test elitism
elites = SelectionStrategy.elitism(test_population, test_fitness, num_elites=2)
print("\nElitism (top 2):")
print(elites)

Available selection methods: ['binary_tournament', 'roulette_wheel', 'rank_selection', 'random_selection']

Binary Tournament Selection:
Parent 1: [ 9 10]
Parent 2: [5 6]

Roulette Wheel Selection:
Parent 1: [7 8]
Parent 2: [5 6]

Rank Selection:
Parent 1: [1 2]
Parent 2: [ 9 10]

Elitism (top 2):
[[ 9 10]
 [ 5  6]]
